In [82]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [83]:
data = pd.read_excel('job_dataset.ods', dtype=str)
target = 'career_level'
x = data.drop(labels= target, axis=1)
y = data[target]
print(y.value_counts())
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=32, stratify=y)

career_level
senior_specialist_or_project_manager      4338
manager_team_leader                       2672
bereichsleiter                             960
director_business_unit_leader               70
specialist                                  30
managing_director_small_medium_company       4
Name: count, dtype: int64


In [84]:
# vectorizer = TfidfVectorizer()
# output = vectorizer.fit_transform(x_train['title'])
# print(output.shape)
# print(vectorizer.vocabulary_)

In [85]:
# def filter_location(location):
#     if ',' in location:
#         return location [-2:]
#     else:
#         return location
    
# data['location'] = data['location'].apply(filter_location)
# data['location']


In [86]:
# encoder = OneHotEncoder()
# output = encoder.fit_transform(x_train[['function']])
# print(output.shape)

In [87]:
x_train.dropna(subset=['description'], inplace=True)
preprocessor = ColumnTransformer(transformers=[
    ('title', TfidfVectorizer(max_features=5000), 'title'),
    ('location', OneHotEncoder(handle_unknown='ignore'), ['location']),
    ('description', TfidfVectorizer(ngram_range=(1,1), stop_words='english', max_features=5000), 'description'),
    ('industry', TfidfVectorizer(stop_words='english', max_features=5000), 'industry')
])

In [88]:
y_train = y_train[x_train.index]

In [89]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

In [90]:
model.fit(x_train, y_train)
y_predict = model.predict(x_test)
print(classification_report(y_test, y_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.82      0.05      0.09       192
         director_business_unit_leader       0.00      0.00      0.00        14
                   manager_team_leader       0.61      0.64      0.62       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.77      0.93      0.84       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.72      1615
                             macro avg       0.37      0.27      0.26      1615
                          weighted avg       0.71      0.72      0.67      1615



c:\Users\caong\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\caong\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\caong\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Em tăng hiệu quả của bài toán classification giảm thời gian chạy của code từ 1p hơn xuống dưới 10s do dùng n_jobs (full core của máy tính), dùng max feature để tập trung vào những features quan trọng